In [1]:
# import os
# os.environ['JAX_PLATFORMS'] = 'cpu'

import os
os.environ['XLA_PYTHON_CLIENT_PREALLOCATE'] = 'false'


# This is required to run multiple processes with JAX.
from multiprocessing import set_start_method
set_start_method('forkserver', force=True)

In [2]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp
from tqdm import tqdm
from pathlib import Path


from config import Config
import data
import train

In [3]:
run_dir = Path("/nas/cee-water/cjgleason/ted/swot-ml/runs/gmm/era5_zstd_20251024_204028")
trainer = train.Trainer.load_last_checkpoint(run_dir)

Loading model from /nas/cee-water/cjgleason/ted/swot-ml/runs/gmm/era5_zstd_20251024_204028/checkpoints/step_100000
Model contains 1,030,584 parameters
Loading trainer from checkpoint step_100000
Logging at /nas/cee-water/cjgleason/ted/swot-ml/runs/gmm/era5_zstd_20251024_204028


In [4]:
cfg = trainer.cfg
cfg.quiet = False
mgr = data.DynamicCacheManager(cfg)
cache_dir = mgr.create_cache('test')

Caches will be stored at: /scratch3/workspace/tlanghorst_umass_edu-swot-ml-zarr2/20251004/_cache2/12f3c9c9b5b17993/<subset>


Loading basins: 100%|██████████| 46/46 [00:00<00:00, 1974.66it/s]

Wrote 0 new basin files to cache.
✅ Cached resources are loaded and ready.


In [5]:
dataset = data.CachedBasinGraphDataset(cfg, cache_dir, 'test')

Calculating training statistics for encoding and normalization...
Loading static attributes...Done!
Loading basin graphs...Done!


Building sampling index: 100%|██████████| 46/46 [00:28<00:00,  1.61it/s]


In [6]:
basin, date, batch = dataset[0]
batch.static.shape

(1410, 438)

In [ ]:
basin, date, batch = dataset[0]

batch.static.shape

In [ ]:
dataset.s_encoding

In [7]:
cfg.num_workers = 12
dataloader = data.CachedBasinGraphDataLoader(cfg, dataset, training=False)

Dataloader using 12 parallel CPU worker(s).


In [8]:
import evaluate
results_file = run_dir / 'results' / 'test_results.parquet'
evaluate.inference.predict_to_parquet(trainer.model, dataloader, results_file, quiet=False)

  0%|          | 0/5377 [01:41<?, ?it/s]


TypeError: CachedBasinGraphDataLoader.denormalize_std() missing 1 required positional argument: 'sigma_norm'

In [21]:
batch.dynamic['era5'].shape

(90, 1410, 10)

In [23]:
trainer.cfg.model_args

STGATArgs(hidden_size=64, dropout=0.3, head='gmm', seq2seq=False, seed=0, name='st_gat', k_hops=3, num_heads=2, return_weights=False, target=['discharge'], seq_length=90, dense_sizes={'era5': 10}, sparse_sizes={}, static_size=438, assim_sizes={})

(1410, 309)

In [12]:
trainer.model

ST_GATransformer(
  head={
    'discharge':
    GMM(
      fc1=Linear(
        weight=f32[128,64],
        bias=f32[128],
        in_features=64,
        out_features=128,
        use_bias=True
      ),
      fc2=Linear(
        weight=f32[300,128],
        bias=f32[300],
        in_features=128,
        out_features=300,
        use_bias=True
      ),
      _eps=1e-05
    )
  },
  target=['discharge'],
  hidden_size=64,
  seq_length=90,
  dense_sources=['era5'],
  sparse_sources=[],
  static_size=438,
  seq2seq=False,
  return_weights=False,
  dense_embedders={
    'era5':
    MLP(
      layers=(
        Linear(
          weight=f32[64,10],
          bias=f32[64],
          in_features=10,
          out_features=64,
          use_bias=True
        ),
        Linear(
          weight=f32[64,64],
          bias=f32[64],
          in_features=64,
          out_features=64,
          use_bias=True
        )
      ),
      activation=<PjitFunction of <function relu at 0x7538bf9b5620>>,
   